# Validate PARTIAL Columns: Salience & Bills Referenced

This notebook validates the two columns that were flagged as PARTIAL in the replication scorecard:
1. **salience** - Google Trends policy salience
2. **bills_referenced** - Bill references per mention

The goal is to determine whether each column is usable for GLMM replication.

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np

# Revamp level1
revamp = pd.read_csv('../../data/output/level1.csv.gz', compression='gzip')
print(f'Revamp: {len(revamp):,} rows, {revamp.shape[1]} columns')

# Legacy dataset
legacy = pd.read_csv('../../data/output/data_legacy_thesis.txt', sep='\t')
print(f'Legacy: {len(legacy):,} rows, {legacy.shape[1]} columns')

C:\Users\kaleb\AppData\Local\Temp\ipykernel_75736\1756802002.py:5: DtypeWarning: Columns (28,29,31,32,33,34,35,36,52,53,54,57,58) have mixed types. Specify dtype option on import or set low_memory=False.
  revamp = pd.read_csv('../../data/output/level1.csv.gz', compression='gzip')


Revamp: 53,892 rows, 63 columns


Legacy: 22,414 rows, 1 columns


## 2. Validate `salience` Column

The legacy thesis used Google Trends data to measure issue salience per policy area. The revamp pipeline has a `1.data_collection/4.policy_salience` module but the data may not have been fully integrated.

In [2]:
# 2a: Check revamp salience
print('=== REVAMP SALIENCE ===')
if 'salience' in revamp.columns:
    non_null = revamp['salience'].notna().sum()
    total = len(revamp)
    print(f'Column present: YES')
    print(f'Non-null: {non_null:,} / {total:,} ({non_null/total*100:.1f}%)')
    print(f'Unique values: {revamp["salience"].dropna().nunique()}')
    print(f'\nValue counts:')
    print(revamp['salience'].value_counts(dropna=False).head(10))
    print(f'\nDescriptive stats:')
    print(revamp['salience'].describe())
else:
    print('Column present: NO')
    # Check for alternatives
    sal_cols = [c for c in revamp.columns if 'salien' in c.lower()]
    print(f'Alternative salience columns: {sal_cols}')

=== REVAMP SALIENCE ===
Column present: YES
Non-null: 14,390 / 53,892 (26.7%)
Unique values: 1

Value counts:
salience
NaN     39502
50.0    14390
Name: count, dtype: int64

Descriptive stats:
count    14390.0
mean        50.0
std          0.0
min         50.0
25%         50.0
50%         50.0
75%         50.0
max         50.0
Name: salience, dtype: float64


In [3]:
# 2b: Check legacy salience columns
print('=== LEGACY SALIENCE COLUMNS ===')
sal_cols = [c for c in legacy.columns if 'salien' in c.lower() or 'saliency' in c.lower()]
print(f'Found {len(sal_cols)} salience-related columns:\n')
for col in sal_cols:
    nn = legacy[col].notna().sum()
    nu = legacy[col].dropna().nunique()
    print(f'  {col}:')
    print(f'    Non-null: {nn:,}, Unique: {nu}')
    print(f'    Sample: {legacy[col].dropna().unique()[:5].tolist()}')
    print()

=== LEGACY SALIENCE COLUMNS ===
Found 1 salience-related columns:



  level1_org_id,level1_variation,level1_granuleId,level1_p1_original,level1_uuid_paragraph,level1_overlap_ids,level1_overlap_count,level1_paragraph_mention_count,level1_mention_index,level1_uuid_mention,level1_p5_cleaned_highlight_index,level1_congress,level1_acronym,level1_prominence,level1_dateIssued_x,level1_packageId_CREC,level1_collectionCode,level1_title_x,level1_collectionName,level1_granuleClass,level1_bookNumber,level1_pagePrefix,level1_subGranuleClass,level1_docClass,level1_lastModified,level1_category,level1_granuleDate,level1_time,level1_rin,level1_legislativeDay,level1_granuleId_count,level1_billnumber_generated,level1_authorityId,level1_p.bioGuideId,level1_memberName,level1_speaker_provided,level1_i.bioGuideId,level1_s.bioGuideId,level1_billVersion,level1_billType,level1_packageId_BILL,level1_billNumber,level1_shortTitle,level1_bill_pred,level1_bill_pred_cat,level1_bill_policy_num,level1_bill_issue_cat,level1_link_generated,level1_weekIssued_CRREC,level1_year_CREC,level1_

    Sample: ['1453.0,Fraternal Order of Police,CREC-2017-05-17-pt1-PgS2980,"Let me express my gratitude to the senior Senator from Minnesota, Ms. Klobuchar, as well as the senior Senators from Connecticut and California--all Democratic colleagues--for being my original cosponsors on the bill. I am also grateful to my Republican colleagues, including Senator Cruz, as well as the junior Senator from North Carolina and the senior Senators from Iowa, Utah, and Nevada, for working with us on this legislation. My friend Congressman Will Hurd on the House side introduced the same bill there, and I am hopeful it will pass sometime today so we can get this to the President\'s desk for his signature without delay. I would also note that the American Law Enforcement Heroes Act is backed by major law enforcement groups across the country, including the Fraternal Order of Police, the Major County Sheriffs of America, the Major City Chiefs Association, and the Veterans of Foreign Wars. I have been g

In [4]:
# 2c: Diagnosis
print('=== SALIENCE DIAGNOSIS ===')
if 'salience' in revamp.columns:
    unique_vals = revamp['salience'].dropna().unique()
    if len(unique_vals) == 1:
        print(f'PROBLEM: Revamp salience has only ONE unique value: {unique_vals[0]}')
        print('This is a constant/placeholder, NOT usable as a predictor.')
        print()
        print('The legacy used Google Trends-derived salience with 57 unique values')
        print('ranging from 1 to ~45, representing weekly search volume by policy area.')
        print()
        print('VERDICT: salience column is NOT USABLE for GLMM Model A.')
        print('Model A (Policy Salience) cannot be replicated without re-collecting')
        print('Google Trends data via the pipeline salience module.')
        salience_status = 'NOT USABLE'
    else:
        print(f'Revamp salience has {len(unique_vals)} unique values - potentially usable')
        salience_status = 'NEEDS FURTHER VALIDATION'
else:
    print('Salience column not present in revamp.')
    salience_status = 'NOT PRESENT'
print(f'\nCheck if issue_area exists for future salience join:')
for col in ['issue_area', 'issue_area_name', 'policy_area']:
    if col in revamp.columns:
        print(f'  {col}: {revamp[col].notna().sum():,} non-null, {revamp[col].nunique()} unique')

=== SALIENCE DIAGNOSIS ===
PROBLEM: Revamp salience has only ONE unique value: 50.0
This is a constant/placeholder, NOT usable as a predictor.

The legacy used Google Trends-derived salience with 57 unique values
ranging from 1 to ~45, representing weekly search volume by policy area.

VERDICT: salience column is NOT USABLE for GLMM Model A.
Model A (Policy Salience) cannot be replicated without re-collecting
Google Trends data via the pipeline salience module.

Check if issue_area exists for future salience join:
  issue_area: 14,390 non-null, 18 unique
  issue_area_name: 14,390 non-null, 18 unique


## 3. Validate `bills_referenced` Column

The legacy thesis used bill sponsorship/cosponsorship data from Congress.gov. The revamp has a `bills_referenced` column counting bill references per speech.

In [5]:
# 3a: Check revamp bills_referenced
print('=== REVAMP BILLS_REFERENCED ===')
if 'bills_referenced' in revamp.columns:
    non_null = revamp['bills_referenced'].notna().sum()
    total = len(revamp)
    non_zero = (revamp['bills_referenced'] != 0).sum()
    print(f'Column present: YES')
    print(f'Non-null: {non_null:,} / {total:,} ({non_null/total*100:.1f}%)')
    print(f'Non-zero: {non_zero:,} / {total:,} ({non_zero/total*100:.1f}%)')
    print(f'\nDescriptive stats:')
    print(revamp['bills_referenced'].describe())
    print(f'\nValue counts (top 15):')
    print(revamp['bills_referenced'].value_counts().head(15))
else:
    print('Column present: NO')
    bill_cols = [c for c in revamp.columns if 'bill' in c.lower()]
    print(f'Alternative bill columns: {bill_cols}')

=== REVAMP BILLS_REFERENCED ===
Column present: YES
Non-null: 53,892 / 53,892 (100.0%)
Non-zero: 13,900 / 53,892 (25.8%)

Descriptive stats:
count    53892.000000
mean         2.392674
std          9.544041
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max        440.000000
Name: bills_referenced, dtype: float64

Value counts (top 15):
bills_referenced
0     39992
4      3769
6      2592
2      1754
8      1726
14     1305
10      876
12      641
20      324
16      157
24      156
18      109
26       55
28       48
22       23
Name: count, dtype: int64


In [6]:
# 3b: Check legacy bill columns
print('=== LEGACY BILL COLUMNS ===')
bill_cols = [c for c in legacy.columns if 'bill' in c.lower()]
print(f'Found {len(bill_cols)} bill-related columns:\n')
for col in ['level1_bills_sponsored', 'level1_bills_cosponsored']:
    if col in legacy.columns:
        nn = legacy[col].notna().sum()
        nu = legacy[col].dropna().nunique()
        print(f'  {col}:')
        print(f'    Non-null: {nn:,}, Unique: {nu}')
        print(f'    Stats: {legacy[col].describe().to_dict()}')
        print()

=== LEGACY BILL COLUMNS ===
Found 1 bill-related columns:



In [7]:
# 3c: Diagnosis
print('=== BILLS_REFERENCED DIAGNOSIS ===')
print('Revamp\'s `bills_referenced` counts how many bills are referenced in each speech.')
print('Legacy\'s `bills_sponsored`/`bills_cosponsored` count politician-level sponsorship activity.')
print()
print('These measure DIFFERENT things:')
print('  - bills_referenced (revamp): speech-level bill reference count')
print('  - bills_sponsored (legacy): politician-level total bill sponsorship')
print()
if 'bills_referenced' in revamp.columns:
    non_zero = (revamp['bills_referenced'] != 0).sum()
    total = len(revamp)
    print(f'Revamp bills_referenced has {non_zero:,}/{total:,} non-zero values ({non_zero/total*100:.1f}%)')
    print('This is USABLE as a control variable but measures something different from legacy.')
    print()
    print('For Model B replication (Group-Politician Linkage):')
    print('  - bills_referenced can serve as a speech-level bill engagement proxy')
    print('  - bills_sponsored/cosponsored would need Congress.gov API re-collection')
    bills_status = 'USABLE (different measure)'
else:
    bills_status = 'NOT PRESENT'
    print('bills_referenced not present.')

=== BILLS_REFERENCED DIAGNOSIS ===
Revamp's `bills_referenced` counts how many bills are referenced in each speech.
Legacy's `bills_sponsored`/`bills_cosponsored` count politician-level sponsorship activity.

These measure DIFFERENT things:
  - bills_referenced (revamp): speech-level bill reference count
  - bills_sponsored (legacy): politician-level total bill sponsorship

Revamp bills_referenced has 13,900/53,892 non-zero values (25.8%)
This is USABLE as a control variable but measures something different from legacy.

For Model B replication (Group-Politician Linkage):
  - bills_referenced can serve as a speech-level bill engagement proxy
  - bills_sponsored/cosponsored would need Congress.gov API re-collection


## 4. Updated Column Status

In [8]:
# Print final status
print('=' * 50)
print('PARTIAL COLUMN VALIDATION')
print('=' * 50)
print()
print(f'salience:         {salience_status}')
print(f'bills_referenced: {bills_status}')
print()
print('IMPACT ON GLMM REPLICATION:')
print('-' * 50)
print('Model A (Policy Salience):       CANNOT REPLICATE')
print('  Reason: salience column is constant (50.0),')
print('  Google Trends data not collected for revamp.')
print('  Workaround: Could use issue_area as categorical')
print('  fixed effect instead of continuous salience.')
print()
print('Model B (Group-Politician):      PARTIALLY REPLICABLE')
print('  Available: seniority, election timing, chamber, party')
print('  Missing: policy_overlap, bills_sponsored/cosponsored')
print('  Available substitute: bills_referenced (speech-level)')
print()
print('Model C (Group Characteristics): FULLY REPLICABLE')
print('  Available: org_age, log_lobbying, policy_scope,')
print('  CATEGORY, MSHIP_STATUS11')
print()
print('UPDATED SCORECARD:')
print(f'  salience:         PARTIAL -> NOT USABLE')
print(f'  bills_referenced: PARTIAL -> USABLE (different measure)')
print(f'  Overall: 15 READY, 1 USABLE (alt measure), 1 NOT USABLE')

PARTIAL COLUMN VALIDATION

salience:         NOT USABLE
bills_referenced: USABLE (different measure)

IMPACT ON GLMM REPLICATION:
--------------------------------------------------
Model A (Policy Salience):       CANNOT REPLICATE
  Reason: salience column is constant (50.0),
  Google Trends data not collected for revamp.
  Workaround: Could use issue_area as categorical
  fixed effect instead of continuous salience.

Model B (Group-Politician):      PARTIALLY REPLICABLE
  Available: seniority, election timing, chamber, party
  Missing: policy_overlap, bills_sponsored/cosponsored
  Available substitute: bills_referenced (speech-level)

Model C (Group Characteristics): FULLY REPLICABLE
  Available: org_age, log_lobbying, policy_scope,
  CATEGORY, MSHIP_STATUS11

UPDATED SCORECARD:
  salience:         PARTIAL -> NOT USABLE
  bills_referenced: PARTIAL -> USABLE (different measure)
  Overall: 15 READY, 1 USABLE (alt measure), 1 NOT USABLE
